# Early engagement and student outcomes

This notebook runs the analysis in order. All of the working code lives in
`src/`, so that each step here is one short call and the logic can be read and
tested on its own. The written findings are in `README.md`.

## 1. Build the analysis table

One row per student per module-presentation. The builder returns the table and
a log of every counting decision, so that no row disappears without a reason
being recorded.

In [1]:
# Run from the repository root regardless of where Jupyter was launched,
# so that `src` imports and the outputs/ paths resolve the same way.
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from src.clean import build_analysis_table

analysis, log = build_analysis_table()
print(log)
print(f"\nTable: {len(analysis):,} rows. Analysis sample: "
      f"{int(analysis['in_analysis_sample'].sum()):,} rows.")

  studentInfo rows                32,593   one row per student per presentation
  after registration join         32,593   one-to-one, no rows gained or lost
  no clicks in days 0-29           4,655   set to 0 clicks, not dropped and not missing
  no weighted mean score           9,376   sat no assessment that carried weight; left missing
  imd_band missing                 1,111   kept in the table; statsmodels drops them from the models
  unregistered before day 0        2,678   no opportunity to click in the window; outside analysis sample
  unregistered during days 0-29    2,389   click window truncated by the withdrawal; outside analysis sample
  analysis sample                 27,526   still registered at day 30

Table: 32,593 rows. Analysis sample: 27,526 rows.


## 2. Why 5,067 records sit outside the analysis sample

Students who unregistered before day 30 had their click window cut short by the
withdrawal itself. Leaving them in would mean partly measuring the outcome with
the exposure. This is the single most consequential choice in the analysis, so
it is worth seeing what it does to the headline rate.

In [2]:
full_rate = analysis['withdrawn'].mean() * 100
sample_rate = analysis.loc[analysis['in_analysis_sample'], 'withdrawn'].mean() * 100
print(f"Withdrawal rate, all records:     {full_rate:.1f}%")
print(f"Withdrawal rate, analysis sample: {sample_rate:.1f}%")

Withdrawal rate, all records:     31.2%
Withdrawal rate, analysis sample: 18.5%


## 3. Outcomes by deprivation band

The main widening participation variable. Students whose band is not recorded
are shown as their own row rather than dropped, because that group turns out to
be distinctive.

In [3]:
from src.describe import outcome_rates_by_imd

outcome_rates_by_imd(analysis)

,students,withdrawn_pct,passed_pct,mean_score,median_early_clicks
imd_band,,,,,
0-10%,2648,21.5,44.0,65.4,146.0
10-20%,2824,19.7,48.1,66.8,157.0
20-30%,2983,21.9,49.9,68.3,166.0
30-40%,2998,18.5,55.4,69.6,177.0
40-50%,2713,18.4,55.9,69.6,166.0
50-60%,2687,17.3,56.7,69.6,183.0
60-70%,2511,18.5,60.1,71.0,197.0
70-80%,2516,17.3,58.9,71.0,196.5
80-90%,2379,16.4,62.8,72.3,196.0


## 4. Is the missing deprivation band random?

The "not recorded" row above has better outcomes than any real band, which is a
warning sign. If the missingness were random it would be spread evenly across
regions.

In [4]:
sample = analysis.loc[analysis['in_analysis_sample']]
missing_by_region = (
    sample.groupby('region', observed=True)['imd_band']
    .apply(lambda s: s.isna().mean() * 100)
    .sort_values(ascending=False)
    .round(1)
)
missing_by_region.head(5).to_frame('percent missing')

,percent missing
region,
North Region,44.0
Ireland,22.6
West Midlands Region,1.8
South Region,1.7
Scotland,0.3


## 5. The two charts

Written to `outputs/` and embedded in the README. Each function returns the
sentence of interpretation that accompanies it.

In [5]:
from pathlib import Path
from src.describe import plot_withdrawal_by_imd, plot_early_clicks

Path('outputs').mkdir(exist_ok=True)
print(plot_withdrawal_by_imd(analysis, Path('outputs/withdrawal_by_imd.png')))
print()
print(plot_early_clicks(analysis, Path('outputs/early_clicks.png')))

Withdrawal is highest among students from the most deprived areas (21.5%) and lowest among those from the least deprived (16.0%), a gap of 5.5 percentage points, though the decline across the middle bands is uneven rather than steady. 1,028 students whose band is not recorded are left out of this chart.

Most students click a few hundred times in the first month, the median being 184, but the spread is wide and 1,197 students (4.3%) record no clicks at all.


## 6. Model 1: withdrawal

Logistic regression, pooled across modules with a fixed effect for each
module-presentation, and standard errors clustered by student. Engagement
enters per 100 clicks so that the odds ratio is readable.

In [6]:
from src.models import prepare, fit, odds_ratio_table, reported_terms

model_sample = prepare(analysis)
withdrawal, withdrawal_data = fit(model_sample, 'withdrawn', 'logit')
odds_ratio_table(withdrawal, reported_terms(withdrawal))

,odds_ratio,ci_low,ci_high,p_value
Intercept,0.143,0.100,0.204,0.000
C(imd_band)[T.10-20%],0.917,0.797,1.054,0.223
C(imd_band)[T.20-30%],1.038,0.905,1.189,0.594
C(imd_band)[T.30-40%],0.835,0.728,0.958,0.010
C(imd_band)[T.40-50%],0.842,0.731,0.969,0.017
C(imd_band)[T.50-60%],0.791,0.684,0.914,0.001
C(imd_band)[T.60-70%],0.864,0.747,0.998,0.047
C(imd_band)[T.70-80%],0.784,0.676,0.908,0.001
C(imd_band)[T.80-90%],0.743,0.639,0.865,0.000
C(imd_band)[T.90-100%],0.719,0.616,0.840,0.000


## 7. What that means in plain rates

An odds ratio is hard to act on. These are recycled predictions, the equivalent
of Stata's `margins`: every student is given each click level in turn, keeping
their real background and module, and the predictions are averaged.

In [7]:
from src.models import predicted_at_click_levels

predicted_at_click_levels(withdrawal, withdrawal_data).assign(
    predicted_pct=lambda d: (d['predicted'] * 100).round(1)
)[['early_clicks', 'predicted_pct']]

,early_clicks,predicted_pct
0,0,23.9
1,100,21.9
2,200,20.0
3,400,16.6
4,800,11.1


## 8. Models 2 and 3: passing and mean score

Note that the pass outcome is 0 for anyone who withdrew, so it combines
continuing and attaining. It is not independent evidence from Model 1.

In [8]:
from src.models import coefficient_table

passing, _ = fit(model_sample, 'passed', 'logit')
score, _ = fit(model_sample, 'mean_score', 'ols')

print("Model 2, odds of passing per 100 clicks:")
print(odds_ratio_table(passing, ['early_clicks_100']).to_string())
print("\nModel 3, marks per 100 clicks:")
print(coefficient_table(score, ['early_clicks_100']).to_string())
print(f"\nModel 3 R-squared: {score.rsquared:.3f}")

Model 2, odds of passing per 100 clicks:
                  odds_ratio  ci_low  ci_high  p_value
early_clicks_100       1.281   1.258    1.304      0.0

Model 3, marks per 100 clicks:
                  coefficient  ci_low  ci_high  p_value
early_clicks_100        1.096   1.013    1.178      0.0

Model 3 R-squared: 0.190


## 9. Does each extra 100 clicks matter equally?

A single linear term assumes it does. Splitting engagement into five equal
groups lets the data answer instead. The reference group is the lowest fifth.

In [9]:
from src.models import engagement_quintile_model

quintiles = engagement_quintile_model(model_sample, 'passed', 'logit')
quintile_terms = [t for t in quintiles.params.index if 'engagement_quintile' in t]
odds_ratio_table(quintiles, quintile_terms)

,odds_ratio,ci_low,ci_high,p_value
C(engagement_quintile)[T.Q2],2.407,2.207,2.625,0.0
C(engagement_quintile)[T.Q3],3.989,3.637,4.374,0.0
C(engagement_quintile)[T.Q4],6.377,5.786,7.028,0.0
C(engagement_quintile)[T.Q5 highest],12.114,10.880,13.489,0.0


The gap between the lowest fifth and the second lowest is the largest single
step. Where support is targeted, the students barely using the site at all are
where the sharpest difference lies.

The findings, and an honest account of what this evidence cannot support, are
written up in `README.md`.